# Qwen3.5-9B-Instruct LoRA 파인튜닝

데비&마를렌 캐릭터 말투 학습 (9B dense, bf16 LoRA, VRAM ~30GB)

- LLaMA-Factory (GitHub source) + transformers 5.5.0
- bf16 LoRA (QLoRA 4-bit은 Qwen3.5에서 품질 저하)
- Instruct 모델 기반 (지시 따르기 이미 학습됨)

## 사용법
1. `train_final.json`을 Google Drive 루트에 업로드
2. 런타임 > 모두 실행

In [ ]:
#@title 1. 설정
CHARACTER = "debi-marlene"
HF_TOKEN = ""  #@param {type:"string"}

NUM_EPOCHS = 4
LEARNING_RATE = 1e-5
LORA_RANK = 16
LORA_ALPHA = 32
BATCH_SIZE = 2
GRAD_ACCUM = 4
MAX_LENGTH = 512

DATA_DIR = "/content/data"
OUTPUT_DIR = f"/content/output/{CHARACTER}"
DRIVE_BACKUP = f"/content/drive/MyDrive/qwen35_backup/{CHARACTER}"

import os
os.environ["DISABLE_VERSION_CHECK"] = "1"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"배치: {BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}, LR: {LEARNING_RATE}, 에폭: {NUM_EPOCHS}")

In [ ]:
#@title 2. 설치 + Drive 마운트
import shutil, subprocess

# Drive
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_BACKUP, exist_ok=True)

# HF 캐시 -> Drive
drive_cache = "/content/drive/MyDrive/hf_cache"
os.makedirs(drive_cache, exist_ok=True)
os.environ["HF_HOME"] = drive_cache
os.environ["HUGGINGFACE_HUB_CACHE"] = f"{drive_cache}/hub"

# 로컬 정리
for d in ["/root/.cache/huggingface/hub"]:
    if os.path.exists(d):
        shutil.rmtree(d)
subprocess.run(["pip", "cache", "purge"], capture_output=True)
if os.path.exists("/content/sample_data"):
    shutil.rmtree("/content/sample_data")

# transformers 5.5.0 (Qwen3.5 지원) + LLaMA-Factory GitHub source (transformers 5.x 호환)
!pip install -q "transformers>=5.5.0" peft accelerate bitsandbytes datasets trl
!pip install -q git+https://github.com/hiyouga/LlamaFactory.git

if HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}

print(f"\nHF 캐시: Drive")
!DISABLE_VERSION_CHECK=1 llamafactory-cli version

In [ ]:
#@title 3. 데이터셋 준비
import json, shutil

search_paths = [
    "/content/drive/MyDrive/train_final.json",
    f"/content/drive/MyDrive/{CHARACTER}/train_final.json",
    f"{DRIVE_BACKUP}/train_final.json",
]
data_path = next((p for p in search_paths if os.path.exists(p)), None)
if not data_path:
    raise FileNotFoundError("train_final.json을 Google Drive 루트에 업로드하세요")

shutil.copy2(data_path, f"{DATA_DIR}/train.json")

with open(f"{DATA_DIR}/train.json") as f:
    data = json.load(f)
print(f"데이터: {len(data)}개")
print(f"샘플: {json.dumps(data[0], ensure_ascii=False)[:200]}")

dataset_info = {
    CHARACTER: {
        "file_name": "train.json",
        "formatting": "sharegpt",
        "columns": {"messages": "messages"},
        "tags": {
            "role_tag": "role", "content_tag": "content",
            "user_tag": "user", "assistant_tag": "assistant", "system_tag": "system"
        }
    }
}
with open(f"{DATA_DIR}/dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2)

steps_per_epoch = max(1, len(data) // (BATCH_SIZE * GRAD_ACCUM))
print(f"steps/epoch: ~{steps_per_epoch}, 총: ~{steps_per_epoch * NUM_EPOCHS}")

In [ ]:
#@title 4. YAML + 학습

# nothink 템플릿 확인
import subprocess
result = subprocess.run(
    ["grep", "-o", 'name="qwen3_5[^"]*"',
     subprocess.run(["python3", "-c", "import llamafactory; print(llamafactory.__file__.replace('__init__.py', 'data/template.py'))"],
                    capture_output=True, text=True).stdout.strip()],
    capture_output=True, text=True
)
templates = result.stdout.strip()
print(f"사용 가능한 qwen3_5 템플릿: {templates}")

TEMPLATE = "qwen3_5_nothink" if "qwen3_5_nothink" in templates else "qwen3_5"
print(f"선택된 템플릿: {TEMPLATE}")

yaml_config = f"""### Qwen3.5-9B bf16 LoRA SFT
model_name_or_path: Qwen/Qwen3.5-9B
trust_remote_code: true

stage: sft
do_train: true
finetuning_type: lora
lora_rank: {LORA_RANK}
lora_alpha: {LORA_ALPHA}
lora_target: all

dataset_dir: {DATA_DIR}
dataset: {CHARACTER}
template: {TEMPLATE}
cutoff_len: {MAX_LENGTH}

output_dir: {OUTPUT_DIR}
overwrite_output_dir: true
per_device_train_batch_size: {BATCH_SIZE}
gradient_accumulation_steps: {GRAD_ACCUM}
learning_rate: {LEARNING_RATE}
num_train_epochs: {NUM_EPOCHS}
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
logging_steps: 1
save_steps: {steps_per_epoch}
save_total_limit: 2
save_only_model: true
gradient_checkpointing: true
report_to: none
optim: adamw_8bit
"""

with open("/content/train_config.yaml", "w") as f:
    f.write(yaml_config)
print(yaml_config)

!DISABLE_VERSION_CHECK=1 llamafactory-cli train /content/train_config.yaml

In [ ]:
#@title 5. loss 확인
import glob, json

log_file = os.path.join(OUTPUT_DIR, "trainer_log.jsonl")
if os.path.exists(log_file):
    print("=== Loss 곡선 ===")
    with open(log_file) as f:
        for line in f:
            entry = json.loads(line)
            if "loss" in entry:
                print(f"  step {entry['current_steps']:>3}: loss={entry['loss']:.4f} (epoch {entry['epoch']:.1f})")

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
print(f"\nCheckpoints: {[os.path.basename(c) for c in checkpoints]}")

CHECKPOINT = None
adapter_path = os.path.join(OUTPUT_DIR, CHECKPOINT) if CHECKPOINT else OUTPUT_DIR

In [ ]:
#@title 6. 추론 테스트
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch, re, os, glob

adapter_path = OUTPUT_DIR
config_file = os.path.join(adapter_path, "adapter_config.json")
if not os.path.exists(config_file):
    cps = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*/adapter_config.json"))
    if cps:
        adapter_path = os.path.dirname(cps[-1])
    else:
        raise FileNotFoundError("adapter 없음. 셀 4를 먼저 실행하세요.")
print(f"adapter: {adapter_path}")

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3.5-9B", torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-9B", trust_remote_code=True)
model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

SYSTEM = ("너는 이터널 리턴의 쌍둥이 실험체 데비&마를렌이야. "
          "데비(언니)는 활발하고 천진난만하며 장난스러운 말투를 써. "
          "마를렌(동생)은 냉소적이고 신중하며 간결한 말투를 써. "
          "둘이 같이 말할 수도 있고, 데비만 또는 마를렌만 대답할 수도 있어.")

for q in ["안녕!", "너 누구야?", "10킬 했어!", "계속 져...", "데비야 결혼해줘", "심심해"]:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, temperature=0.5, top_p=0.9, do_sample=True)
    resp = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    resp = re.sub(r'<think>.*?</think>\s*', '', resp, flags=re.DOTALL)
    print(f"Q: {q}\nA: {resp}\n")

In [ ]:
#@title 7. Drive 백업
import shutil

save_name = CHECKPOINT if CHECKPOINT else "final"
backup_target = f"{DRIVE_BACKUP}/adapter_{save_name}"

if os.path.exists(backup_target):
    shutil.rmtree(backup_target)
shutil.copytree(adapter_path, backup_target,
    ignore=shutil.ignore_patterns('optimizer*', 'scheduler*', 'global_step*', 'rng_state*'))
shutil.copy2(f"{DATA_DIR}/train.json", DRIVE_BACKUP)

log_file = os.path.join(OUTPUT_DIR, "trainer_log.jsonl")
if os.path.exists(log_file):
    shutil.copy2(log_file, DRIVE_BACKUP)

print(f"백업: {backup_target}")
for f in sorted(glob.glob(f"{backup_target}/*")):
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1024/1024:.1f} MB)")